---
# Asssignment 2 - Applied Machine Learning (C1TM1B) Spring 2026
## Image Classification with Artificial Neural Networks
---

- Learning outcomes: 2.1–2.5, 3.1–3.2
- Credits: 1.5
- Gradingscale: Fail (U) or Pass (G)

**This assignment assesses the following learning outcomes:**

**Competence and skills**
- 2.1 implement simple programs in the Python programming language for applied machine learning,
- 2.2 use established software libraries for data processing and machine learning,
- 2.3 use standard data pre-processing techniques,
- 2.4 conduct empirical evaluation of trained classification and regression models, and
- 2.5 carry out design and the implementation of a machine learning pipeline for machine learning.

**Judgement and approach**
- 3.1 choose and apply a machine learning algorithm given problems and data for classification or regression, and
- 3.2 select and apply pre-processing and evaluation methods for different machine learning algorithms

This notebook contains the following sections:
- [1. Prerequisites](#1-prerequisites) which covers setting up the environment.
- [2. Image Preprocessing and Classification](#2-image-preprocessing-and-classification) which introduces basic image preprocessing and classification concepts.
- [3. Assignment Tasks](#3-assignment-tasks) which contains Tasks 1-8 for you to complete as part of the assignment.

**Note!**
- You need to download the following dataset from Canvas and place it in the `notebook/datasets` folder to complete this assignment.
  - `traffic_signs.zip`

---
## 1. Prerequisites

### Dataset

Make sure you have downloaded the `traffic_signs.zip` dataset from Canvas and placed it in the `notebooks/datasets` folder.

### Software

Make sure you have installed the following software on your computer:

- [Visual Studio Code](https://code.visualstudio.com)
- [Git](https://git-scm.com/downloads)
- [Miniconda](https://docs.anaconda.com/miniconda/install/#quick-command-line-install)
- [Python](https://www.python.org) (optional) - comes pre-installed on Linux and Mac
- [Graphviz](https://graphviz.gitlab.io/download)

### Visual Studio Code (VSCode) Extensions

Also make sure you have installed the necessary Visual Studio Code Extensions:

```bash
code --install-extension ms-toolsai.jupyter
code --install-extension ms-python.python
```

### Python Virtual Environment

Finally, make sure you have created a Python virtual environment as below (and selected it as the kernel in the top right of this notebook):

```bash
git clone https://github.com/paga-hb/C1TM1B_2026.git ml
cd ml
conda create -y -p ./.conda python=3.12
conda activate ./.conda
python -m pip install --upgrade pip
pip install ipykernel jupyter numpy pandas matplotlib pydot scikit-learn tensorflow keras_tuner

```

### Python Package Imports

Import the required python packages.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os

# Let's assume no GPU is available (at least for this workshop)
# These two lines have to be executed before importing tensorflow
# If you have a (properly configured) GPU, you can comment out these lines
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # disable GPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # disable Tensorflow C++ log messages

import shutil
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Let's print out if Tensorflow has detected a GPU
print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: []


---
## 2. Image Preprocessing and Classification

One important task within artificial intelligence is using machine learning techniques for **classifying images**. Datasets of images are somewhat different to traditional tabular datasets with rows (data points) and columns (features/attributes) often found in text files, CSV files, or relational databases.

Just as a data point (sample) in a traditional dataset can be multi-dimensional (i.e. have multiple features/attributes), an image sample is multi-dimensional in the sense that it has multiple pixels. Additionally, an image has **spatial structure** associated with it, consisting of a two-dimensional array of pixels (rows and columns), where each pixel usually has three color channels (red, green and blue). With an **8-bit color depth**, each **pixel has a value in the range 0 - 255** in each of the three **color channels (red, green, blue)**. Therefore, a color image actually has a three-dimensional structure, where the first two dimensions are the rows (height) and columns (width), and the third dimension is the three color channels stacked on top of each other. Such images are refered to as RGB (red, green, blue) images. Some images also have a fourth channel - an *alpha* channel - that decides the transparancy/opaqueness of each pixel, and are refered to as RGBA (red, green, blue, alpha) images.

<div style="display: flex; justify-content: space-between: 5px; margin: 20px;">
    <table style="border-collapse: collapse; width: 15%; margin-right: 10px; color: red; float: left; clear: left;">
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
    </table>
    <table style="border-collapse: collapse; width: 15%; margin-right: 10px; color: green;">
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
    </table>
    <table style="border-collapse: collapse; width: 15%; margin-right: 10px; color: blue;">
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
    </table>
</div>

A color image can be converted into a **grayscale image**, where the image colapses into a two-dimensional structure consisting of rows (height) and columns (width) and a **single grayscale color channel**, where each **pixel has a value between 0 (black) and 255 (white)** using an **8-bit color depth**.

<div style="display: flex; justify-content: space-between: 5px; margin: 20px;">
    <table style="border-collapse: collapse; width: 15%; margin-right: 10px; color: #D97706; float: left; clear: left;">
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
        </tr>
        <tr>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="25px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
    </table>
</div>

Therefore, the grayscale image above actually represents a white letter **X** as shown below.

<div style="display: flex; justify-content: space-between: 5px; margin: 20px;">
    <table style="table-layout: fixed; border-collapse: collapse; color: #D97706;">
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
        </tr>
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
        </tr>
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
        </tr>
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
        </tr>
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="black"></td>
            <td width="28px" height="40px" align="middle" vertical-align="middle" bgcolor="white"></td>
        </tr>
    </table>
</div>

If we convert a color image into a grayscale image, and then **flatten** it, we get something that resembles a traditional data point with a number of features/attributes (pixels). When **flattening an image**, we just place each row after each other.

<div style="display: flex; justify-content: space-between: 5px; margin: 20px;">
    <table style="table-layout: fixed; border-collapse: collapse; color: #D97706;">
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">255</td>
        </tr>
    </table>
</div>

Lastly, we can **normalize** the image by **rescaling the pixel values to floats between 0 and 1.0** (i.e. we simply divide each pixel value with 255.0).

<div style="display: flex; justify-content: space-between: 5px; margin: 20px;">
    <table style="table-layout: fixed; border-collapse: collapse; color: #D97706;">
        <tr>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">0</td>
            <td width="28px" height="40px" align="middle" vertical-align="middle">1.0</td>
        </tr>
    </table>
</div>

Now we see that the original 5x5, 8-bit color image has been reduced to a data point with 25 features/attributes (pixels with values between 0 and 1.0), and we already know how to work with such data using traditional machine learning algorithms.

In this assignmed, you will carry-out the pre-processing steps above, to reduce color images into flattened grayscale images with normalized (rescaled) pixel values, before feeding them into a **fully-connected, feed-forward neural network**. There are other neural network architectures that work well (even better) with images without flattening them first (neural networks with **convolutional layers**), but you will **only be using fully-connected, feed-forward layers** (called `Dense` layers in Keras) in this assignment.

Keras, the high-level Tensorflow API, will be used to train neural network models for **classifying images**, i.e. if you feed an image into a trained neural network, it should output which class the image belongs to (e.g. if an image contains an object of a cat, dog or a horse, the network should output the corresponding animal class).

---
## 3. Assignment Tasks

**Note! You may use the Image Classification part of Workshop 3 as an aide when completing this assignment.**

### Task 1 - Load the `traffic_signs` dataset from disk and do initial preprocessing 

1. Unzip the image dataset `traffic_signs.zip` in the `datasets` folder.
   - Your folder/file-structure should look like the following:

   ```
   datasets/
     traffic_signs/
       0/
       1/
       2/
       3/
       4/
       5/
       .../
       39/
       40/
       41/
       42

2. Write some code in the cell below to load the dataset and split it into a training-, validation- and test set.
   - Also do any initial preprocessing such as choosing the batch size, image size, converting the images to grayscale, etc.
   - You can also write code below to do the initial unzipping of `traffic_signs.zip` into the `datasets` folder if you want.

In [42]:
## Write your code here to load a dataset and do initial preprocessing

### Task 2 - Plot sample images from your dataset

1. Use Matplotlib to plot sample images from your dataset (so you know what kind of images you will be classifying).

In [43]:
## Write your code here to plot sample images from your dataset

### Task 3 - Design a model and do any remaning preprocessing

1. Use Keras to design a model for classifying images from your dataset:
   - Choose any desired approach, i.e. the Sequential model, the Functional API, or model subclassing.
2. Depending on the approch you use above (Sequential, etc.), create a model summary (and optionally a model plot with Graphviz).
   - Visualizing the model architecture with summaries/plots can help you in debugging your design. 

In [45]:
## Write your code here to assemble a model (and summarize/plot it)

### Task 4 - Compile and train the model with relevant Keras callbacks

1. Choose relevant Keras Callbacks.
2. Choose an optimizer, loss function and metrics.
3. Compile your model with your optimizer, loss function, metrics and callbacks.
4. Train your model on your traing set, and validate it on your validation set during training.

In [46]:
## Write your code here to create callbacks, an optimizer, a loss function, and metrics, and to train your model

### Task 5 - Evaluate the Model on the test set and visualize your Model's training statistics

1. Evaluate the performance of your model on the test set.
2. Plot your model's learning curves.
   - Use the training statistics in the `History` dictionary returned by the `fit()` method.
   - You can also visualize training statistics via TensorBoard.

In [48]:
## Write your code here to evaluate your model on the test set, and to plot your model's learning curves

### Task 6 - Improve the Model (the topics below are covered in Chapters 10-11 in the course book)

1. After creating an initial baseline of your model, try improving your model by experimenting with some of the topics below:
   - Number of hidden layers, and number of neurons per hidden layer.
   - Number of iterations (epochs), using early stopping or not.
   - Batch size.
   - Learning rate and learning rate scheduling.
   - Optimizer (sgd, rmsprop, adam, etc.).
   - Regularization (L1/L2, dropout, etc.).
   - Activation functions (relu, selu, etc.).
   - Kernel initialisers (golorot_uniform, he_uniform, etc.).
   - Batch Normalization.
2. Choose at least two topics above, and write down your comments regarding the outcome (if it improved the model) and why.

**Note!** You may also try using the Keras Tuner to automate the hyperparameter search.

In [49]:
# Write down your comments here, including what topics you chose, the resulting outcome, including why you think it improved the model.

### Task 7 - Q&A

1. Answer the questions below (wite your answers as comments underneath the questions).

In [52]:
# Question 1: How can you detect overfitting by plotting the training loss and validation loss in the same graph?


# Question 2: How can you detect underfitting by plotting the training loss and validation loss in the same graph?


# Question 3: What should the plot of the training loss and validation loss (the "learning curves") idealy look like?


# Question 4: What influence does the number of layers, and number of neurons per layer, have on underfitting/overfitting? Why?


# Question 5: What influence does the number of training samples have on underfitting/overfitting? Why?


# Question 6: What does early stopping do, and how does it help with model performance?


# Question 7: How do you think the batch size affects model performance? Why?


# Question 8: What can you say about the learning rate, and learning rate scheduling? How does a too high/low learning rate affect performacne? Why?


# Question 9: What is regularization, and how does it help with model performance?


# Question 10: What is the purpose of kernel initializers? Why not just initialize all weights to 0?


# Question 11: Does 95% accuracy on the training/validation set for a classification task always paint an accurate picture of model performance? Why?


### Task 8 - Submission

1. Make sure you have completed Tasks 1-7 above.
2. Make sure you can execute each cell in this notebook without any errors.
3. Submit this notebook via Canvas before the deadline.